<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myProspects_Upside_Analysis_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance
!pip install ta

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29482 sha256=b9a3899b33df590c2ffb286abe445ffec5482e7aaee8073fd52bde0eff3ecdd7
  Stored in directory: /root/.cache/pip/wheels/e3/3a/ee/4955a26c90a4b7deb6d725dc8ec7b8604a7aef44e43a2e8af7
Successfully built ta


In [2]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
import os
project_path = '/content/drive/My Drive/quantvesting'
os.chdir(project_path)

import warnings
warnings.filterwarnings('ignore')

from google.colab import data_table

Mounted at /content/drive


In [3]:
# Import the function from the script
import myProspects_Upside_Analysis_v2

all = 1

# Call the function to get the DataFrame
df_prospects_features = myProspects_Upside_Analysis_v2.get_prospects_features_v2(all)
cols = ['Symbol', 'Close', 'FTT',  'Dev%_200', 'Dev%_PE', 'Spread%', 'Conviction', 'Cyclical','RSI_14', 'RSP', 'Today%', 'FTT%', 'ATH%','Gained%', 'Criteria', 'Strategy', 'Sales_Grwth%', 'Profit_Grwth%', 'Category', 'InFolio', 'MCap', 'MBQ', 'ROE%', 'CFO_2_EBITDA%', 'ROE%/PE']
df_prospects_features = df_prospects_features[cols].drop_duplicates()

Run date time (IST): 2026-11-06 09:55:08
-------------------
get_prospects_features
qualified stocks: 287
with latest results: 287
still star stocks: 112


In [4]:
cols = ['Symbol', 'FTT', 'Dev%_200', 'Dev%_PE', 'Spread%', 'Conviction','Cyclical', 'RSI_14', 'RSP', 'FTT%', 'ATH%','Gained%', 'CumlRnk','ROE%/PE', 'Criteria', 'Strategy', 'Category', 'InFolio']

tmp_df = df_prospects_features
list_x40 = list(tmp_df[tmp_df['MBQ'].str.contains('X40', na=False)]['Symbol'].values)
list_x5k = list(tmp_df[tmp_df['MBQ'].str.contains('X5K', na=False)]['Symbol'].values)
list_x40n = list(tmp_df[tmp_df['MBQ'].str.contains('X40N', na=False)]['Symbol'].values)
list_hc = list(tmp_df[tmp_df['Conviction'].isin(['X-LC', 'H-LC', 'X-MC', 'X-SC'])]['Symbol'].values)
list_ox40 = list(tmp_df[tmp_df['MBQ'].str.contains('OX40', na=False)]['Symbol'].values)
list_ox40n = list(tmp_df[tmp_df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)
#len(set(list_x40 + list_x5k) - set( list_x40n))
stock_list = list(set(list(set(list_x40 + list_x5k + list_hc)- set(list_ox40 + list_ox40n))))

In [5]:
df_tmp = df_prospects_features.copy()

# value ranks (ascending better for deviation from PE and 200DMA)
df_tmp['Rnk_Dev%_200'] = df_tmp['Dev%_200'].rank(method='min', ascending=True)
df_tmp['Rnk_Dev%_PE'] = df_tmp['Dev%_PE'].rank(method='min', ascending=True)
df_tmp['Rnk_ROE%/PE'] = df_tmp['ROE%/PE'].rank(method='max', ascending=False)

# growth ranks
df_tmp['Rnk_Sales_Grwth%'] = df_tmp['Sales_Grwth%'].rank(method='max', ascending=False)
df_tmp['Rnk_Profit_Grwth%'] = df_tmp['Profit_Grwth%'].rank(method='max', ascending=False)

# quality ranks
df_tmp['Rnk_FTT%'] = df_tmp['FTT%'].rank(method='max', ascending=False)
df_tmp['Rnk_MCap'] = df_tmp['MCap'].rank(method='max', ascending=False)
df_tmp['Rnk_ROE%'] = df_tmp['ROE%'].rank(method='max', ascending=False)
df_tmp['Rnk_CFO_2_EBITDA%'] = df_tmp['CFO_2_EBITDA%'].rank(method='max', ascending=False)

# optional Momentum ranks
df_tmp['Rnk_RSI_14'] = df_tmp['RSI_14'].rank(method='max', ascending=False)
df_tmp['Rnk_Gained%'] = df_tmp['Gained%'].rank(method='min', ascending=True)

# category scores
df_tmp['Rank_Value'] = df_tmp[['Rnk_Dev%_200', 'Rnk_Dev%_PE', 'Rnk_ROE%/PE']].sum(axis=1)
df_tmp['Rank_Growth'] = df_tmp[['Rnk_Sales_Grwth%', 'Rnk_Profit_Grwth%']].sum(axis=1)
df_tmp['Rank_Quality'] = df_tmp[['Rnk_FTT%', 'Rnk_MCap', 'Rnk_ROE%', 'Rnk_CFO_2_EBITDA%']].sum(axis=1)
df_tmp['Rank_Momentum'] = df_tmp[['Rnk_Gained%', 'Rnk_RSI_14']].sum(axis=1)

# final cumulative score (you can weigh categories if needed)
df_tmp['Ovrl_Rank'] = df_tmp[['Rank_Value', 'Rank_Growth', 'Rank_Quality', 'Rank_Momentum']].sum(axis=1)

# category-wise rank
df_tmp['Cat_Rank'] = df_tmp.groupby('Conviction')['Ovrl_Rank'].rank(ascending=True, method='min')
priority_map = {'X-LC': 0,'H-LC': 1,'X-MC': 2,'X-SC': 3,'M-LC': 4,'H-MC': 5,'H-SC': 6,'L-LC': 7,'M-MC': 8,'M-SC': 9,'L-MC': 10,'L-SC': 11}
df_tmp['Conviction_Priority'] = df_tmp['Conviction'].map(priority_map)
df_tmp = df_tmp.sort_values(by=['Conviction_Priority', 'Cat_Rank']).reset_index(drop=True)

# final rank
df_tmp['CumlRnk'] = df_tmp.index + 1

# drop intermediate rank columns (optional)
drop_cols = [col for col in df_tmp.columns if col.startswith('Rnk_')]
df_tmp.drop(columns=drop_cols, inplace=True)

# reassign to main dataframe
df_prospects_features = df_tmp[cols]

# display
data_table.DataTable(df_prospects_features.sort_values(by='CumlRnk'), include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
0,TMPV,600.00,-12.75,-85.99,17.23,X-LC,DC,59.0,56.45,105.90,292.06,11.56,1,58.2,XY24,NTT,AUTO,1.0
1,NESTLEIND,1540.00,-7.62,-20.68,8.16,X-LC,NC,48.0,21.60,23.02,23.02,7.80,2,1.2,XY25,ATH,FMCG,1.0
2,INFY,1972.00,1.98,-37.74,8.43,X-LC,NC,65.0,96.86,66.13,56.57,23.23,3,2.0,X40,NTT,IT,1.0
3,HINDUNILVR,2922.00,-6.92,-43.44,11.43,X-LC,NC,64.0,67.25,48.08,47.46,8.74,4,1.0,XY25,NTT,FMCG,1.0
4,TCS,4208.34,6.73,-38.28,6.47,X-LC,NC,66.0,97.56,65.73,65.73,29.46,5,2.8,X40,ATH,IT,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282,VIPIND,489.00,-22.35,NaN,16.85,NaN,NC,29.0,6.62,99.92,202.56,0.06,283,NaN,OX40N,NTT,MISC,NaN
283,WESTLIFE,987.24,28.05,NaN,15.29,NaN,DC,67.0,88.50,51.17,51.17,56.77,284,NaN,AR,ATH,RESTAURANTS,NaN
284,WHIRLPOOL,2270.00,-1.70,NaN,1.36,NaN,DC,48.0,23.00,178.90,222.62,13.40,285,NaN,XR,NTT,DURABLES,1.0
285,WSTCSTPAPR,940.45,38.14,NaN,48.43,NaN,DC,46.0,49.83,24.46,24.46,99.16,286,NaN,XR,ATH,PAPER,NaN


In [6]:
# most interesting sectors
df_tmp = df_prospects_features.groupby('Category').agg(
    Count=('CumlRnk', 'count'),
    Total=('CumlRnk', 'sum'),
    Average=('CumlRnk', 'mean')
).reset_index()
df_tmp['Average'] = round(df_tmp['Average'],2)
df_tmp = df_tmp[df_tmp['Count'] > 3]
df_tmp.sort_values(by='Average')

,Category,Count,Total,Average
3,AUTO,12,940,78.33
23,INSURANCE,8,696,87.00
16,FMCG,19,1718,90.42
27,METALS,5,471,94.20
7,CEMENT,4,390,97.50
12,ELECTRICAL,9,956,106.22
33,PHARMA,18,1977,109.83
31,PAINTS,5,599,119.80
20,HEALTHCARE,8,972,121.50
21,HOTELS,4,533,133.25


In [7]:
# module to print date timestamp
def read_csv(csv_file_name):
  csv_file_path = os.path.join(project_path, 'data', csv_file_name)
  df = pd.read_csv(csv_file_path)
  return df

mypps_df = read_csv('myProspectsScrips.csv')

# Suppose this is your updated DataFrame with new cuml_rnk values
cols1 = ['Symbol', 'CumlRnk']
df_tmp = df_prospects_features[cols1]

# Merge the two DataFrames on 'symbol', keeping the CSV structure
mypps_df = mypps_df.merge(df_tmp[['Symbol', 'CumlRnk']], on='Symbol', how='left', suffixes=('', '_new'))

# Replace 'cuml_rnk' with the new value only if it exists in df
mypps_df['CumlRnk'] = mypps_df['CumlRnk_new'].combine_first(mypps_df['CumlRnk'])

# Drop the helper column
mypps_df.drop(columns=['CumlRnk_new'], inplace=True)

# Save back to CSV
#mypps_df.to_csv(project_path+'/data/myProspectsScrips.csv', index=False)

In [8]:
# High quality (X40 + X40N + X5K + X-LC + H-LC + X-MC + X-SC) prospects
df_tmp = df_prospects_features[df_prospects_features['Symbol'].isin(stock_list)]
df_tmp = df_tmp[(df_tmp['FTT%']>20) & (df_tmp['Dev%_PE']<0)]
data_table.DataTable(df_tmp.sort_values(by = 'CumlRnk', ascending=True), include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
0,TMPV,600.00,-12.75,-85.99,17.23,X-LC,DC,59.0,56.45,105.90,292.06,11.56,1,58.2,XY24,NTT,AUTO,1.0
1,NESTLEIND,1540.00,-7.62,-20.68,8.16,X-LC,NC,48.0,21.60,23.02,23.02,7.80,2,1.2,XY25,ATH,FMCG,1.0
2,INFY,1972.00,1.98,-37.74,8.43,X-LC,NC,65.0,96.86,66.13,56.57,23.23,3,2.0,X40,NTT,IT,1.0
3,HINDUNILVR,2922.00,-6.92,-43.44,11.43,X-LC,NC,64.0,67.25,48.08,47.46,8.74,4,1.0,XY25,NTT,FMCG,1.0
4,TCS,4208.34,6.73,-38.28,6.47,X-LC,NC,66.0,97.56,65.73,65.73,29.46,5,2.8,X40,ATH,IT,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,RECLTD,446.00,-10.12,-11.71,10.94,M-MC,NC,52.0,49.13,49.92,93.93,5.08,161,4.1,XY25,NTT,FINANCE,1.0
162,IDBI,160.44,-7.13,-41.40,5.12,M-MC,NC,45.0,14.98,103.24,103.24,28.36,163,1.6,X5K,ATH,BANKS,NaN
186,IRB,37.85,-14.75,-31.82,15.70,M-SC,DC,50.0,27.87,121.99,121.99,8.39,187,0.2,X5K,ATH,INFRA,NaN
204,VEDL,772.14,-37.71,-12.67,41.15,L-MC,DC,58.0,43.55,198.70,198.70,14.58,205,7.5,NaN,NaN,MINING,NaN


In [9]:
# X-LC, H-LC & X-MC
df_tmp = df_prospects_features[df_prospects_features['Conviction'].isin(['X-LC','H-LC','X-MC'])].sort_values(by = 'ATH%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
0,TMPV,600.00,-12.75,-85.99,17.23,X-LC,DC,59.0,56.45,105.90,292.06,11.56,1,58.2,XY24,NTT,AUTO,1.0
61,BAJAJHFL,181.50,-3.94,-39.11,5.23,X-MC,NC,53.0,41.46,122.29,122.29,11.71,62,0.5,X40N,ATH,FINANCE,1.0
54,COLPAL,3628.04,-14.42,-23.20,14.40,X-MC,NC,48.0,23.34,117.30,117.30,6.17,55,2.5,XY25,ATH,FMCG,1.0
55,HAVELLS,2052.62,-12.72,-41.25,13.16,X-MC,SC,50.0,51.22,92.92,92.92,5.50,56,0.5,X40,ATH,ELECTRICAL,1.0
18,BAJAJFINSV,2758.00,-0.85,-18.01,7.28,X-LC,NC,65.0,69.69,50.44,74.41,12.44,19,0.5,X40,BTT,FINANCE,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,ICICIBANK,1519.30,12.71,2.11,3.66,X-LC,NC,75.0,93.38,0.00,0.00,27.05,32,0.8,X40,ATH,BANKS,NaN
29,ADANIPORTS,1583.00,15.21,20.63,8.03,X-LC,NC,66.0,80.14,-17.33,0.00,47.50,30,0.5,XY24,NTT,MISC,NaN
32,KOTAKBANK,470.65,17.37,4.86,11.82,X-LC,NC,69.0,85.02,0.00,0.00,33.41,33,0.5,X40,ATH,BANKS,NaN
50,DIVISLAB,10197.50,38.07,37.74,28.34,H-LC,NC,72.0,77.35,0.00,0.00,75.91,51,0.2,AR,ATH,PHARMA,NaN


In [10]:
# x24/x25
df_tmp = df_prospects_features[df_prospects_features['Criteria'].isin(['XY24', 'X25'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
237,EASEMYTRIP,26.40,-14.29,NaN,17.36,NaN,DC,55.0,63.41,340.00,468.50,12.57,238,NaN,XY24,NTT,TRAVEL,1.0
235,DIGITIDE,267.99,-32.17,NaN,26.48,NaN,NC,31.0,5.57,329.47,329.47,3.06,236,NaN,XY24,ATH,IT,NaN
241,GREENPANEL,537.00,-22.25,NaN,25.49,NaN,NC,52.0,14.29,269.35,311.41,16.78,242,NaN,XY24,NTT,MISC,1.0
82,ACC,3906.00,-11.06,-45.85,15.48,X-SC,DC,59.0,66.20,219.64,124.65,8.63,83,0.9,XY24,BTT,CEMENT,1.0
80,AWL,485.00,-11.97,-57.56,7.89,X-SC,NC,38.0,13.59,189.97,400.20,2.08,81,0.6,XY24,NTT,FMCG,1.0
112,IRCTC,1269.00,-11.72,-52.31,12.90,H-MC,DC,54.0,53.66,175.90,144.22,5.44,113,1.3,XY24,BTT,TRAVEL,1.0
269,SHALBY,327.00,-22.21,NaN,20.19,NaN,NC,38.0,10.80,171.19,166.78,6.34,270,NaN,XY24,NTT,HEALTHCARE,1.0
261,REPCOHOME,880.00,-12.38,NaN,13.94,NaN,NC,51.0,48.78,168.54,149.54,7.92,262,NaN,XY24,NTT,FINANCE,1.0
145,BANDHANBNK,400.00,-8.61,37.21,9.35,H-SC,NC,49.0,36.93,148.68,335.49,19.76,146,0.3,XY24,NTT,BANKS,1.0
267,SFL,1287.00,-15.32,NaN,7.07,NaN,NC,29.0,4.88,146.32,276.95,12.51,268,NaN,XY24,NTT,MISC,1.0


In [11]:
# X-MC and X-SC prospects
df_tmp = df_prospects_features[df_prospects_features['Conviction'].str.startswith('X-', na=False)]
df_tmp = df_tmp[~df_tmp['Conviction'].isin(['X-LC'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
82,ACC,3906.00,-11.06,-45.85,15.48,X-SC,DC,59.0,66.20,219.64,124.65,8.63,83,0.9,XY24,BTT,CEMENT,1.0
80,AWL,485.00,-11.97,-57.56,7.89,X-SC,NC,38.0,13.59,189.97,400.20,2.08,81,0.6,XY24,NTT,FMCG,1.0
77,PGHH,17484.41,-23.84,-61.49,24.40,X-SC,NC,49.0,56.79,149.87,149.87,5.40,78,4.0,X40,ATH,FMCG,1.0
61,BAJAJHFL,181.50,-3.94,-39.11,5.23,X-MC,NC,53.0,41.46,122.29,122.29,11.71,62,0.5,X40N,ATH,FINANCE,1.0
54,COLPAL,3628.04,-14.42,-23.20,14.40,X-MC,NC,48.0,23.34,117.30,117.30,6.17,55,2.5,XY25,ATH,FMCG,1.0
85,HONAUT,58192.79,-11.16,-39.76,7.04,X-SC,SC,34.0,9.76,95.87,95.87,13.18,86,0.3,X40N,ATH,ELECTRICAL,1.0
56,DABUR,735.00,-11.58,-35.86,12.63,X-MC,NC,51.0,35.19,93.32,69.83,6.26,57,0.5,XY24,BTT,FMCG,1.0
55,HAVELLS,2052.62,-12.72,-41.25,13.16,X-MC,SC,50.0,51.22,92.92,92.92,5.50,56,0.5,X40,ATH,ELECTRICAL,1.0
87,UBL,2211.69,-12.22,-29.52,14.78,X-SC,NC,55.0,62.37,81.52,81.52,7.23,88,0.1,X40N,ATH,FMCG,NaN
90,LALPATHLAB,3545.00,19.60,-2.83,16.81,X-SC,NC,56.0,49.48,80.41,3.28,51.04,91,0.4,X40N,NTT,HEALTHCARE,NaN


In [12]:
# H-MC and H-SC prospects
df_tmp = df_prospects_features[df_prospects_features['Conviction'].str.startswith('H-', na=False)]
df_tmp = df_tmp[~df_tmp['Conviction'].isin(['H-LC'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
137,METROPOLIS,3102.00,9.25,-0.54,7.59,H-SC,NC,58.0,62.72,442.88,49.54,35.75,138,0.2,XR,NTT,HEALTHCARE,NaN
112,IRCTC,1269.00,-11.72,-52.31,12.90,H-MC,DC,54.0,53.66,175.90,144.22,5.44,113,1.3,XY24,BTT,TRAVEL,1.0
136,PIIND,4686.44,-35.92,-40.64,23.15,H-SC,SC,25.0,1.39,168.86,168.86,0.00,137,0.5,NaN,ATH,CHEMICALS,NaN
145,BANDHANBNK,400.00,-8.61,37.21,9.35,H-SC,NC,49.0,36.93,148.68,335.49,19.76,146,0.3,XY24,NTT,BANKS,1.0
133,TATAELXSI,9161.00,-2.80,-35.84,13.51,H-SC,NC,68.0,99.30,133.46,159.92,30.97,134,0.6,OX40N,NTT,IT,1.0
130,VINATIORGA,2299.51,-23.56,-57.83,18.75,H-SC,SC,35.0,7.32,128.94,128.94,5.32,131,0.8,OX40N,ATH,CHEMICALS,NaN
128,BAYERCROP,6805.00,-21.61,-42.84,18.92,H-SC,SC,34.0,21.08,106.85,99.68,1.23,129,1.2,OX40N,NTT,CHEMICALS,NaN
138,DEEPAKNTR,3075.77,-5.56,-29.52,4.84,H-SC,SC,47.0,34.15,101.54,101.54,19.14,139,0.4,AR,ATH,CHEMICALS,NaN
113,ASTRAL,2402.45,-17.05,-34.25,7.24,H-MC,SC,31.0,11.85,93.59,93.59,0.00,114,0.2,AR,ATH,PLASTICS,NaN
129,NATCOPHARM,1576.63,-11.41,-33.16,14.61,H-SC,NC,59.0,44.25,92.19,92.19,4.96,130,1.6,AR,ATH,PHARMA,NaN


In [13]:
# x40
df_tmp = df_prospects_features[df_prospects_features['Criteria'].isin(['X40'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
224,BATAINDIA,2096.00,-6.20,NaN,7.61,NaN,SC,53.0,66.55,225.72,212.04,10.99,225,NaN,X40,NTT,FOOTWEAR,1.0
77,PGHH,17484.41,-23.84,-61.49,24.40,X-SC,NC,49.0,56.79,149.87,149.87,5.40,78,4.0,X40,ATH,FMCG,1.0
264,SANOFI,7552.00,0.28,NaN,10.04,NaN,NC,68.0,91.29,127.45,183.04,22.41,265,NaN,X40,NTT,PHARMA,NaN
55,HAVELLS,2052.62,-12.72,-41.25,13.16,X-MC,SC,50.0,51.22,92.92,92.92,5.50,56,0.5,X40,ATH,ELECTRICAL,1.0
2,INFY,1972.00,1.98,-37.74,8.43,X-LC,NC,65.0,96.86,66.13,56.57,23.23,3,2.0,X40,NTT,IT,1.0
4,TCS,4208.34,6.73,-38.28,6.47,X-LC,NC,66.0,97.56,65.73,65.73,29.46,5,2.8,X40,ATH,IT,1.0
5,ITC,452.00,-1.52,-27.49,7.49,X-LC,NC,68.0,73.17,60.48,64.73,10.45,6,1.6,X40,NTT,FMCG,1.0
79,PFIZER,6042.10,-14.98,-30.78,16.37,X-SC,NC,49.0,45.30,57.19,57.19,6.41,80,0.8,X40,ATH,PHARMA,NaN
18,BAJAJFINSV,2758.00,-0.85,-18.01,7.28,X-LC,NC,65.0,69.69,50.44,74.41,12.44,19,0.5,X40,BTT,FINANCE,NaN
17,HDFCLIFE,811.41,-7.53,-30.50,10.02,X-LC,NC,60.0,54.01,47.23,47.23,7.30,18,0.2,X40,ATH,INSURANCE,1.0


In [14]:
# x40n
df_tmp = df_prospects_features[df_prospects_features['Criteria'].isin(['X40N'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
260,RELAXO,1176.00,-12.96,NaN,13.86,NaN,SC,49.0,45.64,305.73,382.56,23.21,261,NaN,X40N,NTT,FOOTWEAR,1.0
276,TEAMLEASE,3487.00,-21.93,NaN,15.72,NaN,NC,32.0,11.50,249.75,429.66,0.00,277,NaN,X40N,NTT,MISC,NaN
61,BAJAJHFL,181.50,-3.94,-39.11,5.23,X-MC,NC,53.0,41.46,122.29,122.29,11.71,62,0.5,X40N,ATH,FINANCE,1.0
85,HONAUT,58192.79,-11.16,-39.76,7.04,X-SC,SC,34.0,9.76,95.87,95.87,13.18,86,0.3,X40N,ATH,ELECTRICAL,1.0
87,UBL,2211.69,-12.22,-29.52,14.78,X-SC,NC,55.0,62.37,81.52,81.52,7.23,88,0.1,X40N,ATH,FMCG,NaN
90,LALPATHLAB,3545.00,19.60,-2.83,16.81,X-SC,NC,56.0,49.48,80.41,3.28,51.04,91,0.4,X40N,NTT,HEALTHCARE,NaN
59,GODREJCP,1452.16,-15.79,-24.07,16.51,X-MC,NC,51.0,36.24,70.74,70.74,4.60,60,0.4,X40N,ATH,FMCG,1.0
83,ASTRAZEN,10427.12,-17.02,-8.95,21.91,X-SC,NC,61.0,86.06,60.15,60.15,13.84,84,0.2,X40N,ATH,PHARMA,NaN
10,DMART,5391.45,-13.06,-29.94,16.34,X-LC,NC,52.0,28.57,57.23,57.23,9.42,11,0.2,X40N,ATH,FMCG,1.0
11,VBL,668.62,-4.50,-27.82,8.28,X-LC,NC,57.0,42.86,53.53,53.53,14.46,12,0.4,X40N,ATH,FMCG,1.0


In [15]:
# xr, sr, ar
df_tmp = df_prospects_features[df_prospects_features['Criteria'].isin(['XR','SR', 'AR'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
223,BAJAJHIND,442.93,4.17,NaN,12.33,NaN,DC,47.0,10.10,2262.29,2262.29,25.59,224,NaN,XR,ATH,SUGAR,NaN
217,ALOKINDS,79.81,-30.17,NaN,32.97,NaN,DC,52.0,79.09,852.39,852.39,18.87,218,NaN,AR,ATH,TEXTILES,NaN
210,CENTRALBK,183.18,-7.52,-51.78,9.33,L-SC,NC,55.0,53.31,500.20,500.20,6.23,211,2.0,XR,ATH,BANKS,NaN
137,METROPOLIS,3102.00,9.25,-0.54,7.59,H-SC,NC,58.0,62.72,442.88,49.54,35.75,138,0.2,XR,NTT,HEALTHCARE,NaN
263,ROUTE,2170.75,-10.01,NaN,13.85,NaN,NC,54.0,74.22,379.56,379.56,11.16,264,NaN,SR,ATH,IT,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52,ZYDUSLIFE,1285.01,15.46,25.29,11.87,H-LC,SC,57.0,65.16,7.01,7.01,39.66,53,0.8,AR,ATH,PHARMA,1.0
53,GRASIM,3380.50,7.37,53.57,4.43,H-LC,DC,67.0,81.88,5.44,5.44,27.06,54,0.1,AR,ATH,TEXTILES,NaN
173,SONACOMS,915.25,37.98,1.94,27.59,M-MC,DC,65.0,88.85,1.58,1.58,103.97,174,0.2,AR,ATH,AUTO,NaN
42,JSWSTEEL,1355.60,7.68,-59.94,1.64,H-LC,DC,64.0,76.66,1.12,1.12,24.92,43,0.8,AR,ATH,STEEL,NaN


In [16]:
# conviction-wise prospects count
df_prospects_features['Conviction'].value_counts()

,count
Conviction,
X-LC,35
M-SC,25
X-MC,23
H-SC,21
M-MC,21
H-LC,19
X-SC,18
M-LC,17
H-MC,16
